# Cryptographic Attack

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import hashlib
import random
import string
import time
import threading
from collections import Counter, defaultdict
import itertools
import math

class CryptographicAttackFramework:
    def __init__(self):
        self.wordlist = self._generate_common_passwords()
        self.hash_cache = {}
        self.attack_results = {}
        self.timing_data = []

    def _generate_common_passwords(self):
        common = ['password', '123456', 'admin', 'root', 'user', 'test', 'guest']
        patterns = ['password123', 'admin2023', 'user123', 'test123']
        years = [str(i) for i in range(2020, 2025)]

        wordlist = common + patterns
        for word in common:
            wordlist.extend([word + year for year in years])
            wordlist.extend([word.upper(), word.capitalize()])

        return list(set(wordlist))

    def frequency_analysis(self, ciphertext):
        freq = Counter(ciphertext.upper())
        total = sum(freq.values())
        frequencies = {char: count/total for char, count in freq.items()}

        english_freq = {'E': 0.127, 'T': 0.091, 'A': 0.082, 'O': 0.075, 'I': 0.070,
                       'N': 0.067, 'S': 0.063, 'H': 0.061, 'R': 0.060}

        chi_squared = sum((frequencies.get(char, 0) - expected)**2 / expected
                         for char, expected in english_freq.items())

        return {
            'frequencies': frequencies,
            'chi_squared': chi_squared,
            'likely_english': chi_squared < 50,
            'entropy': self._calculate_entropy(ciphertext)
        }

    def _calculate_entropy(self, text):
        freq = Counter(text)
        length = len(text)
        return -sum((count/length) * math.log2(count/length) for count in freq.values())

    def caesar_crack(self, ciphertext):
        results = []
        for shift in range(26):
            decrypted = self._caesar_decrypt(ciphertext, shift)
            analysis = self.frequency_analysis(decrypted)
            results.append({
                'shift': shift,
                'plaintext': decrypted,
                'score': analysis['chi_squared'],
                'likely_solution': analysis['likely_english']
            })

        return sorted(results, key=lambda x: x['score'])[:3]

    def _caesar_decrypt(self, text, shift):
        result = ""
        for char in text:
            if char.isalpha():
                ascii_offset = 65 if char.isupper() else 97
                result += chr((ord(char) - ascii_offset - shift) % 26 + ascii_offset)
            else:
                result += char
        return result

    def vigenere_crack(self, ciphertext, max_key_length=10):
        best_results = []

        for key_len in range(1, max_key_length + 1):
            key = self._find_vigenere_key(ciphertext, key_len)
            plaintext = self._vigenere_decrypt(ciphertext, key)
            score = self.frequency_analysis(plaintext)['chi_squared']

            best_results.append({
                'key_length': key_len,
                'key': key,
                'plaintext': plaintext,
                'score': score
            })

        return sorted(best_results, key=lambda x: x['score'])[:3]

    def _find_vigenere_key(self, ciphertext, key_length):
        key = []
        clean_text = ''.join(c for c in ciphertext.upper() if c.isalpha())

        for i in range(key_length):
            column = ''.join(clean_text[j] for j in range(i, len(clean_text), key_length))
            best_shift = self.caesar_crack(column)[0]['shift']
            key.append(chr(best_shift + ord('A')))

        return ''.join(key)

    def _vigenere_decrypt(self, ciphertext, key):
        result = ""
        key_index = 0

        for char in ciphertext:
            if char.isalpha():
                shift = ord(key[key_index % len(key)]) - ord('A')
                ascii_offset = 65 if char.isupper() else 97
                result += chr((ord(char) - ascii_offset - shift) % 26 + ascii_offset)
                key_index += 1
            else:
                result += char
        return result

    def hash_attack(self, target_hash, hash_type='md5', attack_type='dictionary'):
        start_time = time.time()

        if attack_type == 'dictionary':
            result = self._dictionary_attack(target_hash, hash_type)
        elif attack_type == 'brute_force':
            result = self._brute_force_attack(target_hash, hash_type, max_length=6)
        else:
            result = self._hybrid_attack(target_hash, hash_type)

        end_time = time.time()

        if result:
            result['time_taken'] = end_time - start_time
            result['attempts_per_second'] = result.get('attempts', 0) / (end_time - start_time)

        return result

    def _dictionary_attack(self, target_hash, hash_type):
        hash_func = getattr(hashlib, hash_type)
        attempts = 0

        for password in self.wordlist:
            attempts += 1
            test_hash = hash_func(password.encode()).hexdigest()

            if test_hash == target_hash:
                return {
                    'success': True,
                    'password': password,
                    'attempts': attempts,
                    'method': 'dictionary'
                }

        return {'success': False, 'attempts': attempts, 'method': 'dictionary'}

    def _brute_force_attack(self, target_hash, hash_type, max_length):
        hash_func = getattr(hashlib, hash_type)
        attempts = 0
        charset = string.ascii_lowercase + string.digits

        for length in range(1, max_length + 1):
            for password_tuple in itertools.product(charset, repeat=length):
                attempts += 1
                password = ''.join(password_tuple)
                test_hash = hash_func(password.encode()).hexdigest()

                if test_hash == target_hash:
                    return {
                        'success': True,
                        'password': password,
                        'attempts': attempts,
                        'method': 'brute_force'
                    }

                if attempts > 100000:
                    break

        return {'success': False, 'attempts': attempts, 'method': 'brute_force'}

    def _hybrid_attack(self, target_hash, hash_type):
        hash_func = getattr(hashlib, hash_type)
        attempts = 0

        for base_word in self.wordlist[:50]:
            for suffix in ['', '1', '123', '!', '2023', '2024']:
                for prefix in ['', '1', 'a']:
                    attempts += 1
                    password = prefix + base_word + suffix
                    test_hash = hash_func(password.encode()).hexdigest()

                    if test_hash == target_hash:
                        return {
                            'success': True,
                            'password': password,
                            'attempts': attempts,
                            'method': 'hybrid'
                        }

        return {'success': False, 'attempts': attempts, 'method': 'hybrid'}

    def timing_attack_simulation(self, correct_password, max_attempts=1000):
        timing_data = []

        for attempt in range(max_attempts):
            guess = self._generate_guess(correct_password, attempt)
            start = time.perf_counter()
            is_correct = self._vulnerable_compare(guess, correct_password)
            end = time.perf_counter()

            timing_data.append({
                'guess': guess,
                'time': end - start,
                'correct': is_correct,
                'common_prefix_length': self._common_prefix_length(guess, correct_password)
            })

        return self._analyze_timing_data(timing_data)

    def _generate_guess(self, target, attempt):
        if attempt < len(target):
            return target[:attempt] + chr(random.randint(97, 122)) + 'x' * (len(target) - attempt - 1)
        else:
            return ''.join(random.choices(string.ascii_lowercase, k=len(target)))

    def _vulnerable_compare(self, guess, correct):
        for i, (g, c) in enumerate(zip(guess, correct)):
            time.sleep(0.00001 * (i + 1))
            if g != c:
                return False
        return len(guess) == len(correct)

    def _common_prefix_length(self, s1, s2):
        for i, (c1, c2) in enumerate(zip(s1, s2)):
            if c1 != c2:
                return i
        return min(len(s1), len(s2))

    def _analyze_timing_data(self, timing_data):
        by_prefix_length = defaultdict(list)

        for data in timing_data:
            by_prefix_length[data['common_prefix_length']].append(data['time'])

        avg_times = {length: sum(times)/len(times)
                    for length, times in by_prefix_length.items()}

        return {
            'timing_correlation': max(avg_times.values()) / min(avg_times.values()) if avg_times else 1,
            'vulnerable_to_timing': max(avg_times.values()) / min(avg_times.values()) > 1.2 if avg_times else False,
            'average_times_by_prefix': avg_times
        }

framework = CryptographicAttackFramework()
test_hash = hashlib.md5(b"password123").hexdigest()
result = framework.hash_attack(test_hash, 'md5', 'dictionary')
print(f"Attack result: {result}")

cipher = "WKLV LV D WHVW"
caesar_result = framework.caesar_crack(cipher)
print(f"Caesar crack: {caesar_result[0]}")

timing_result = framework.timing_attack_simulation("secret", 100)
print(f"Timing attack: {timing_result}")